# 06d · 브라운 마팅게일과 탈출 문제 (Brownian Martingales and the Exit Problem)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Karlin & Taylor 2e §7.5 (마팅게일 방법으로 BM 범함수) · Ross 12e §10.2 (BM 의 도박꾼 파산), §10.5 (드리프트 BM 의 최댓값) · Shreve II §3.3.4 (BM 의 마팅게일 성질), §3.6 (지수 마팅게일 + 선택정지) · Lawler 2e §5.3 (선택정지), §8.8 (드리프트 BM 의 도달 확률) · G&S 4e §12.5 (optional stopping), §12.7 (연속시간 마팅게일) · Durrett 3e §5.3–5.4 (이산 배경) |
| 선수 노트북 | 06c (반사 원리·도달시간·격자 편향), 05b (이산 마팅게일로 푼 도박꾼 파산), 05a (지수 마팅게일 $e^{\theta S_n}/\cosh^n\theta$) |
| 예상 소요 | 90분 |
| 상태 | draft |

05b 에서 $S_n$ 과 $S_n^2-n$ 두 마팅게일에 선택정지정리를 걸어 도박꾼 파산을 풀었다. 이 노트북은 같은 일을 연속시간에서 한다: $B_t$, $B_t^2-t$, 그리고 지수 마팅게일 $e^{\theta B_t-\theta^2t/2}$ 가 **세 개의 열쇠**이고, 구간 $(-a,b)$ 탈출 확률 $a/(a+b)$, 평균 탈출시간 $ab$, 드리프트가 있을 때의 탈출 확률, 한쪽 장벽의 $P(T_b<\infty)=e^{-2|\mu|b/\sigma^2}$, 라플라스 변환 $E[e^{-\alpha T_b}]=e^{-b\sqrt{2\alpha}}$ 가 전부 몇 줄 계산으로 나온다. 시뮬레이션 쪽에서는 06c 의 격자 편향이 양쪽 장벽에서도 그대로 나타나는 것을 보고, 브라운 다리 교차 확률로 지우는 방법을 익힌다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.stats import norm
from scipy.integrate import quad
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.brownian import brownian_paths
from spkit.plots import plot_paths, plot_hist_vs_pdf

SEED, rng = rng_for("06d")
setup_plots()
N_PATHS = scale(20_000)          # 탈출 시뮬레이션 경로 수; FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
N_CURVE = scale(4_000)           # 드리프트별 탈출 확률 곡선용 (점당)
N_LAPLACE = scale(1_000_000)     # 라플라스 변환용 정확 표본 수
N_BINS = 10 if fast() else 20    # 조건부 평균 검사의 분위수 bin 수
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}, N_CURVE={N_CURVE}, N_LAPLACE={N_LAPLACE}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답해 보세요. 답은 접힌 블록에 있습니다.

**Q1.** (06c) 반사 원리의 식과 $E[M_1]$ 은? ($M_t=\max_{s\le t}B_s$)

<details><summary>정답</summary>

$P(M_t\ge a)=2P(B_t\ge a)$ ($a>0$). 따라서 $M_1\overset{d}{=}|B_1|$ 이고 $E[M_1]=\sqrt{2/\pi}\approx0.798$.

</details>

**Q2.** (06c) 도달시간 $T_a$ 의 평균과 중앙값은?

<details><summary>정답</summary>

평균 $\infty$ (밀도 꼬리가 $t^{-3/2}$, 생존함수 꼬리가 $t^{-1/2}$); 중앙값은 $P(T_a\le t)=2(1-\Phi(a/\sqrt t))=1/2$ 에서 $t=a^2/z_{0.75}^2\approx2.198a^2$. 오늘 라플라스 변환 $E[e^{-\alpha T_a}]=e^{-a\sqrt{2\alpha}}$ 의 $\alpha=0$ 에서의 기울기 $-\infty$ 가 바로 $E T_a=\infty$ 다.

</details>

**Q3.** (06c) 격자 최댓값 편향의 크기와 두 가지 처방은?

<details><summary>정답</summary>

격자에서만 최댓값을 보면 장벽을 $\beta\sqrt{\Delta t}$, $\beta=0.5826$ 만큼 밖으로 민 것과 같다 (편향 $\propto\sqrt{\Delta t}$). 처방: (i) $\Delta t$ 를 100 배 줄여 편향을 10 배 줄이거나, (ii) 격자 사이를 브라운 다리로 보고 구간별 최댓값을 정확히 표본추출한다. 오늘은 (ii) 를 양쪽 장벽으로 확장한다.

</details>

**Q4.** (06b) 이차 변동 $Q_n(T)$ 의 극한과 매끄러운 함수의 경우는?

<details><summary>정답</summary>

$Q_n(T)\to T$ ($L^2$, 부분열 a.s.); 매끄러운 함수는 0. $B_t^2-t$ 가 마팅게일인 것은 "$B_t^2$ 이 평균적으로 $t$ 만큼 자란다"는 같은 사실의 조건부 버전이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **연속시간 마팅게일**(continuous-time martingale): 여과(filtration) $\mathcal F_t=\sigma(B_u:u\le t)$ 에 대해 $M_t$ 가 $\mathcal F_t$-가측(적합)이고 $E|M_t|<\infty$, 그리고 $s\le t$ 에서 $E[M_t\mid\mathcal F_s]=M_s$.
- **정지시간**(stopping time) $\tau$: 모든 $t$ 에서 $\{\tau\le t\}\in\mathcal F_t$ — "지금까지 본 것만으로 멈출지 결정된다". 오늘의 두 주인공: 구간 **탈출시간**(exit time) $\tau=\inf\{t:B_t\notin(-a,b)\}$ ($a,b>0$) 과 **도달시간**(hitting time) $T_b=\inf\{t:B_t=b\}$.
- **세 브라운 마팅게일**: $B_t$; $B_t^2-t$; 지수(Wald) 마팅게일 $Y^\theta_t=\exp(\theta B_t-\theta^2t/2)$. 드리프트 BM $X_t=\mu t+\sigma B_t$ 에 대해서는 $\exp\big(\theta X_t-(\theta\mu+\theta^2\sigma^2/2)t\big)$ 가 마팅게일이고, 특히 $\theta=-2\mu/\sigma^2$ 를 고르면 시간항이 사라져 $e^{-2\mu X_t/\sigma^2}$ 가 (시간항 없는) 마팅게일이다.
- **선택정지정리**(optional stopping theorem, 연속시간; 오늘 쓰는 형태): 마팅게일 $M$ 과 정지시간 $\tau$ 에 대해 다음 중 하나가 성립하면 $E[M_\tau]=M_0$: (i) $\tau\le K$ 유계; (ii) $\tau<\infty$ a.s. 이고 정지 과정 $(M_{t\wedge\tau})_t$ 가 유계; (iii) $(M_{t\wedge\tau})_t$ 가 균등적분가능. (Lawler 2e §5.3, G&S 4e §12.5/§12.7; Durrett 3e §5.3 의 이산판을 연속시간으로.) 증명의 뼈대는 항상 같다: 유계 정지시간 $t\wedge\tau$ 에서 $E[M_{t\wedge\tau}]=M_0$ 은 거저 얻고, $t\to\infty$ 에서 극한과 기댓값을 교환할 근거가 (ii)/(iii) 다.
- **탈출 시뮬레이션의 다리 교차 보정**(bridge-crossing correction): 한 스텝에서 $x\to y$ 로 이동했고 둘 다 $(-a,b)$ 안에 있어도, 스텝 도중에 $b$ 를 넘었을 확률은 06c 의 브라운 다리 최댓값 CDF 를 $z=b$ 에서 평가한
  $$p_{\uparrow}=\exp\!\Big(-\frac{2(b-x)(b-y)}{\sigma^2\Delta t}\Big),\qquad p_{\downarrow}=\exp\!\Big(-\frac{2(a+x)(a+y)}{\sigma^2\Delta t}\Big)$$
  이다. 이 확률로 탈출을 선언하면 탈출 **확률**은 (한 스텝에 양쪽을 다 건드리는 $O(e^{-c/\Delta t})$ 사건을 빼면) 정확해지고, 탈출 **시각**은 스텝 중점 $(k+\tfrac12)\Delta t$ 를 쓰므로 오차 $\le\Delta t/2$ 다.

### 2.2 정리 1 — 세 브라운 마팅게일 (Shreve II §3.3.4, Karlin & Taylor 2e §7.5)

$B_t$, $B_t^2-t$, $Y^\theta_t=\exp(\theta B_t-\theta^2t/2)$ 는 $\mathcal F_t$-마팅게일이다. 드리프트 판: $X_t=\mu t+\sigma B_t$ 에 대해 $e^{-2\mu X_t/\sigma^2}$ 도 마팅게일이다.

가정이 왜 필요한가:
- **독립 증분**: $E[g(B_t-B_s)\mid\mathcal F_s]=E[g(B_t-B_s)]$ 로 조건부 기대를 보통 기대로 바꾸는 유일한 도구다. 이것이 없으면 $\mathcal F_s$ 가 미래 증분에 대해 무언가 말해 주므로 $E[B_t\mid\mathcal F_s]\ne B_s$ 일 수 있다.
- **가우스 증분**: 정규 mgf $E[e^{\theta(B_t-B_s)}]=e^{\theta^2(t-s)/2}$ 가 지수 마팅게일의 시간항 $e^{-\theta^2t/2}$ 를 정확히 상쇄한다. 증분 분포가 다르면 보정항도 다르다 — 05a 의 $\pm1$ 걸음에서는 $e^{\theta S_n}/\cosh^n\theta$ 였다.
- **적분가능성**: $E|Y^\theta_t|=E[Y^\theta_t]=1<\infty$, $E[B_t^2]=t<\infty$. 마팅게일 정의의 $E|M_t|<\infty$ 는 조건부 기대가 정의되기 위한 최소 조건이다.

<details><summary>증명</summary>

$s<t$ 로 두고 $D=B_t-B_s\sim N(0,t-s)$ 라 하자. 독립 증분에 의해 $D$ 는 $\mathcal F_s$ 와 독립이고, $B_s$ 는 $\mathcal F_s$-가측이다.

**(1) $B_t$.** $E[B_t\mid\mathcal F_s]=E[B_s+D\mid\mathcal F_s]=B_s+E[D]=B_s$.

**(2) $B_t^2-t$.** $E[B_t^2\mid\mathcal F_s]=E[(B_s+D)^2\mid\mathcal F_s]=B_s^2+2B_sE[D]+E[D^2]=B_s^2+(t-s)$. 양변에서 $t$ 를 빼면 $E[B_t^2-t\mid\mathcal F_s]=B_s^2-s$.

**(3) $Y^\theta_t$.** $E[e^{\theta B_t}\mid\mathcal F_s]=e^{\theta B_s}E[e^{\theta D}]=e^{\theta B_s}e^{\theta^2(t-s)/2}$ (정규 mgf). 양변에 $e^{-\theta^2t/2}$ 를 곱하면 $E[Y^\theta_t\mid\mathcal F_s]=e^{\theta B_s-\theta^2s/2}=Y^\theta_s$.

**드리프트 판.** $\theta X_t-(\theta\mu+\theta^2\sigma^2/2)t=\theta\sigma B_t+\theta\mu t-\theta\mu t-\theta^2\sigma^2t/2=(\theta\sigma)B_t-(\theta\sigma)^2t/2$ 이므로 이것은 (3) 의 $\theta\sigma$ 판, 즉 $Y^{\theta\sigma}_t$ 다. $\theta=-2\mu/\sigma^2$ 를 넣으면 시간항의 계수가 $\theta\mu+\theta^2\sigma^2/2=-2\mu^2/\sigma^2+2\mu^2/\sigma^2=0$ 이 되어 $e^{-2\mu X_t/\sigma^2}$ 자체가 마팅게일이다. 적분가능성은 위 "가정" 항목에서 확인했다. $\square$

</details>

### 2.3 정리 2 — 구간 탈출: 확률 $a/(a+b)$, 평균시간 $ab$ (도박꾼 파산의 4 번째 풀이; Ross 12e §10.2, Karlin & Taylor 2e §7.5)

$B_0=0$, $\tau=\inf\{t:B_t\in\{-a,b\}\}$ ($a,b>0$). 그러면 $\tau<\infty$ a.s., $E\tau<\infty$ 이고
$$P(B_\tau=b)=\frac{a}{a+b},\qquad E[\tau]=ab.$$
일반 시작점 $x\in(-a,b)$: $P_x(B_\tau=b)=\dfrac{x+a}{a+b}$, $E_x\tau=(x+a)(b-x)$. 일반 $\sigma$ ($X_t=\sigma B_t$): 확률은 불변, $E\tau=ab/\sigma^2$.

00a 에서 시뮬레이션으로, 01c 에서 기본행렬 $N=(I-Q)^{-1}$ 로, 05b 에서 이산 마팅게일 $S_n$, $S_n^2-n$ 으로 풀었던 도박꾼 파산의 네 번째 풀이다: $i\leftrightarrow x+a$, $N\leftrightarrow a+b$ 로 읽으면 $i/N$ 과 $i(N-i)$ 가 그대로 나온다.

가정이 왜 필요한가:
- **정지 과정 $B_{t\wedge\tau}$ 의 유계성** ($|B_{t\wedge\tau}|\le\max(a,b)$): $B$ 에 대한 선택정지 조건 (ii). 장벽이 한쪽뿐이면($a=\infty$) 유계가 아니고 실제로 $E[B_{T_b}]=b\ne0$ 이다 (E3, 함정 1).
- **$E\tau<\infty$**: $B_t^2-t$ 쪽에서 $E[t\wedge\tau]\to E\tau$ 의 단조수렴은 공짜지만, 결론이 유한한 수의 등식이 되려면 $E\tau<\infty$ 가 있어야 한다. $T_b$ 는 $ET_b=\infty$ 라 같은 논법이 "$\infty=\infty$" 로 끝난다.
- **경로 연속성**: $B_\tau\in\{-a,b\}$ 정확히 — overshoot 가 없다. 05b 의 이산 걸음은 정수 장벽 덕분에 우연히 같은 성질을 가졌고, 걸음 크기가 정수가 아니면 overshoot 항이 생긴다.

<details><summary>증명</summary>

**(0) $\tau<\infty$ a.s., $E\tau<\infty$.** $L=a+b$ 라 하자. 단위 시간 구간 $[k,k+1]$ 마다 $A_k=\{|B_{k+1}-B_k|>L\}$ 은 확률 $p=2(1-\Phi(L))>0$ 인 독립 사건이고, $A_k$ 가 일어나면 구간 폭이 $L$ 인 $(-a,b)$ 를 그 안에서 반드시 벗어난다. 따라서 $\{\tau>k\}\subset A_0^c\cap\dots\cap A_{k-1}^c$ 이고 $P(\tau>k)\le(1-p)^k$: 기하 꼬리이므로 $\tau<\infty$ a.s. 이고 모든 모멘트가 유한하다.

**(1) 확률.** $B_{t\wedge\tau}$ 는 $[-a,b]$ 안에 갇힌 유계 마팅게일이고 $\tau<\infty$ a.s. 이므로 선택정지 (ii) 로 $E[B_\tau]=B_0=0$. $p=P(B_\tau=b)$ 라 하면 $B_\tau$ 는 $b$ 아니면 $-a$ 이므로 $bp-a(1-p)=0$, 즉 $p=a/(a+b)$.

**(2) 평균시간.** $M_t=B_t^2-t$ 와 유계 정지시간 $t\wedge\tau$ (조건 (i)) 에 선택정지를 걸면 $E[B_{t\wedge\tau}^2]=E[t\wedge\tau]$. $t\to\infty$: 좌변은 $B_{t\wedge\tau}^2\le\max(a,b)^2$ 로 유계수렴정리에 의해 $E[B_\tau^2]=b^2p+a^2(1-p)=\dfrac{ab^2+a^2b}{a+b}=ab$; 우변은 단조수렴정리로 $E\tau$. 따라서 $E\tau=ab$ (유한, (0) 과 일치).

**일반 $x$.** $B_t-x$ 는 0 에서 시작하는 BM 이고 장벽은 $-(x+a)$, $b-x$ 이므로 $a\to x+a$, $b\to b-x$ 를 대입한다. **일반 $\sigma$.** $\sigma B_t$ 가 $(-a,b)$ 를 벗어나는 것은 $B_t$ 가 $(-a/\sigma,b/\sigma)$ 를 벗어나는 것과 같은 사건·같은 시각이므로 확률은 $(a/\sigma)/((a+b)/\sigma)=a/(a+b)$, 시간은 $(a/\sigma)(b/\sigma)=ab/\sigma^2$. $\square$

</details>

### 2.4 정리 3 — 드리프트 BM 의 탈출, 한쪽 도달, 라플라스 변환 (Ross 12e §10.5, Lawler 2e §8.8, Shreve II §3.6)

$X_t=\mu t+\sigma B_t$ ($\mu\ne0$), $\tau$ 는 $(-a,b)$ 탈출시간. 그러면
$$P(X_\tau=b)=\frac{1-e^{2\mu a/\sigma^2}}{e^{-2\mu b/\sigma^2}-e^{2\mu a/\sigma^2}}=\frac{e^{2\mu a/\sigma^2}-1}{e^{2\mu a/\sigma^2}-e^{-2\mu b/\sigma^2}},\qquad E\tau=\frac{b\,p-a(1-p)}{\mu}\quad(p=P(X_\tau=b)).$$
한쪽 장벽 ($a\to\infty$): $\mu<0$ 이면 $P(T_b<\infty)=e^{2\mu b/\sigma^2}=e^{-2|\mu|b/\sigma^2}$, 즉 $\sup_tX_t\sim\mathrm{Exp}(2|\mu|/\sigma^2)$; $\mu>0$ 이면 $T_b<\infty$ a.s. 이고 $E T_b=b/\mu$. $\mu=0$ 의 라플라스 변환: $E[e^{-\alpha T_b}]=e^{-b\sqrt{2\alpha}}$ ($\alpha>0$).

가정이 왜 필요한가:
- **$e^{-2\mu X_{t\wedge\tau}/\sigma^2}$ 의 유계성**: 구간 안에서 $X$ 가 유계이므로 지수도 유계 — 선택정지 (ii).
- **$E\tau<\infty$** (정리 2 (0) 과 같은 기하 꼬리 논법): $X_t-\mu t=\sigma B_t$ 에 선택정지를 걸어 $E[X_\tau]=\mu E\tau$ — 월드 항등식(Wald's identity)의 연속판.
- **한쪽 극한 $a\to\infty$**: $\mu<0$ 이면 $e^{2\mu a/\sigma^2}\to0$ 으로 극한이 깔끔하고, 탈출 사건 $\{X_\tau=b\}$ 가 $a\uparrow\infty$ 에서 $\{T_b<\infty\}$ 로 단조 수렴한다.
- **라플라스**: $\theta=\sqrt{2\alpha}>0$ 에서 $Y^\theta_{t\wedge T_b}\le e^{\theta b}$ 유계 → 선택정지 (ii). $T_b<\infty$ a.s. 는 06c (반사 원리) 에서 왔다. $\theta<0$ 을 골랐다면 $Y^\theta_{t\wedge T_b}$ 가 아래로 무한히 커질 수 있어 논법이 깨진다.

<details><summary>증명</summary>

**탈출 확률.** $e^{-2\mu X_{t\wedge\tau}/\sigma^2}$ 는 유계 마팅게일, $\tau<\infty$ a.s. 이므로 $1=E[e^{-2\mu X_\tau/\sigma^2}]=p\,e^{-2\mu b/\sigma^2}+(1-p)e^{2\mu a/\sigma^2}$. $p$ 에 대해 풀면 $p=\dfrac{1-e^{2\mu a/\sigma^2}}{e^{-2\mu b/\sigma^2}-e^{2\mu a/\sigma^2}}$; 분자·분모에 $-1$ 을 곱하면 둘째 꼴.

**평균 탈출시간.** $\sigma B_{t\wedge\tau}=X_{t\wedge\tau}-\mu(t\wedge\tau)$ 에서 $E[X_{t\wedge\tau}]=\mu E[t\wedge\tau]$; $t\to\infty$ 에서 좌변은 유계수렴, 우변은 단조수렴으로 $E[X_\tau]=\mu E\tau$, 즉 $bp-a(1-p)=\mu E\tau$.

**한쪽 장벽.** $\mu<0$: $a\to\infty$ 에서 $e^{2\mu a/\sigma^2}\to0$ 이므로 $p\to e^{2\mu b/\sigma^2}$; 사건 $\{X_\tau=b\}$ 는 $a$ 에 대해 증가하며 $\{T_b<\infty\}=\{\sup_tX_t\ge b\}$ 로 수렴하므로 $P(\sup X\ge b)=e^{-2|\mu|b/\sigma^2}$: $\sup X\sim\mathrm{Exp}(2|\mu|/\sigma^2)$, 평균 $\sigma^2/(2|\mu|)$. $\mu>0$: $p\to1$ 이고 $E\tau=(bp-a(1-p))/\mu$ 에서 $a(1-p)\to0$ ($1-p\sim e^{-2\mu a/\sigma^2}$ 가 지수적으로 작다) 이므로 $E T_b=\lim E\tau=b/\mu$.

**라플라스 변환.** $\theta>0$ 에 대해 $Y^\theta_{t\wedge T_b}=\exp(\theta B_{t\wedge T_b}-\theta^2(t\wedge T_b)/2)\le e^{\theta b}$ 유계, $T_b<\infty$ a.s. → $E[e^{\theta B_{T_b}-\theta^2T_b/2}]=1$. $B_{T_b}=b$ 이므로 $E[e^{-\theta^2T_b/2}]=e^{-\theta b}$; $\alpha=\theta^2/2$, 즉 $\theta=\sqrt{2\alpha}$ 를 넣으면 $E[e^{-\alpha T_b}]=e^{-b\sqrt{2\alpha}}$. 검산: $\alpha\downarrow0$ 에서 값 1 ($T_b<\infty$ a.s.), $\frac{d}{d\alpha}\big|_{0^+}=-E[T_b]=-b/\sqrt{2\alpha}\big|_{0^+}=-\infty$ — 06c 의 $ET_b=\infty$ 와 일치. $\square$

</details>

### 2.5 직관

05b 에서 $S_n$ 과 $S_n^2-n$ 으로 도박꾼 파산을 풀었듯이, 연속시간에서는 $B_t$ 와 $B_t^2-t$ 가 같은 역할을 하고 답도 같은 꼴이다: $i/N\leftrightarrow(x+a)/(a+b)$, $i(N-i)\leftrightarrow(x+a)(b-x)$. "공정한 게임은 어디서 멈춰도 공정하다"($E[B_\tau]=0$) 가 확률을, "분산은 시간만큼 쌓인다"($E[B_\tau^2]=E\tau$) 가 시간을 준다.

지수 마팅게일은 **드리프트를 지우는 환율**이다. $\mu>0$ 이면 $X$ 는 위로 가려 하지만, $e^{-2\mu X/\sigma^2}$ 는 위로 갈수록 작아지고 아래로 갈수록 커져서 평균이 정확히 보존된다. 그래서 $1=p\,e^{-2\mu b/\sigma^2}+(1-p)e^{2\mu a/\sigma^2}$ 한 줄이 탈출 확률을 준다. 05b 의 $(q/p)^{S_n}$ 이 정확히 같은 환율이다 (E1(d)).

시뮬레이션 쪽: 탈출은 06c 의 최댓값 문제를 양쪽 장벽으로 확장한 것이라 격자 편향도 같다 — 장벽이 $0.58\sqrt{\Delta t}$ 씩 밖으로 밀려 $E\tau\approx(a+\delta)(b+\delta)$ 로 커진다. 다리 교차 확률로 보정하면 탈출 확률은 정확, 탈출 시각은 $\Delta t/2$ 이내.

### 2.6 함정 (traps)

- $E[B_\tau]=0$ 을 쓰려면 정지 과정의 유계성/적분가능성이 필요하다. 한쪽 장벽 $T_b$ 는 $ET_b=\infty$ 이고 $B_{t\wedge T_b}$ 가 아래로 무한하므로 $E[B_{T_b}]=b\ne0$ (E3).
- $E\tau=ab$ 는 $\sigma=1$ 일 때. 일반 $\sigma$ 에서는 $ab/\sigma^2$ (탈출 확률은 $\sigma$ 와 무관).
- 드리프트 공식의 부호 검산: $\mu>0$ 이면 $P(X_\tau=b)>a/(a+b)$ 이어야 한다 ($\mu=0.25$, $a=1$, $b=2$: $0.5065>0.3333$). 부호가 뒤집힌 식을 써도 $\mu=0$ 극한은 똑같이 $1/3$ 이라 알아채기 어렵다.
- 다리 보정 없는 격자 탈출은 $\Delta t=0.005$ 에서도 $E\tau$ 를 약 6% (9 SE) 과대추정한다. 'ok' 가 안 나오면 난수보다 보정 여부를 먼저 확인.
- 다리 교차 확률은 $\sigma^2\Delta t$ 로 나눈다: $\exp(-2(b-x)(b-y)/(\sigma^2\Delta t))$. $\sigma\ne1$ 에서 빠뜨리기 쉽다.
- $e^{\theta B_t-\theta^2t/2}$ 의 표본평균은 1 이지만 분산 $e^{\theta^2t}-1$ 이 $t$ 에 지수적이다: $\theta=1$, $t=1$ 이면 se $\approx0.009$ ($N=20000$) 지만 $t=5$ 면 $0.086$ — 07a/07e 의 측도 변환(measure change) 추정량이 불안정한 이유.
- 탈출 시뮬레이션의 시간 지평 $t_{\max}$: $P(\tau>t)\approx Ce^{-\lambda_1t}$, $\lambda_1=\pi^2/(2(a+b)^2)$. $a+b=3$, $t_{\max}=20$ 이면 경로당 $\approx1.7\times10^{-5}$ → $N=20000$ 에서 미탈출 경로가 실제로 나온다. $t_{\max}=40$ ($3\times10^{-10}$) 으로 잡는다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요 — 특히 $E[B_\tau]=0$ 이 왜 "공정한 게임은 어디서 멈춰도 공정하다"는 말인지, 그리고 그것이 왜 한쪽 장벽 $T_b$ 에서는 거짓인지 두 문장으로.)

## 3. Predict — 코드를 돌리기 전에 예측

코드를 돌리기 전에 아래 값을 먼저 적어 보세요. 손계산 힌트: $a/(a+b)$, $ab$, $\frac{e^{2\mu a}-1}{e^{2\mu a}-e^{-2\mu b}}$, $(bp-a(1-p))/\mu$, $e^{2\mu b}$, $e^{-b\sqrt{2\alpha}}$.

In [ ]:
predictions = {
    # B_0 = 0, 장벽 -a = -1, b = 2: P(2 를 -1 보다 먼저 침) 은?
    "p_hit_b_first": None,
    # 같은 설정에서 평균 탈출시간 E[tau] 는?
    "e_exit_time": None,
    # 드리프트 mu = 0.25, sigma = 1, 같은 장벽: P(X_tau = 2) 는? (1/3 보다 커야 하나 작아야 하나?)
    "p_hit_b_first_drift": None,
    # 같은 드리프트 설정에서 E[tau] 는?
    "e_exit_time_drift": None,
    # mu = -0.5, sigma = 1, 한쪽 장벽 b = 1: P(sup_t X_t >= 1) 은?
    "p_ever_hit_1_negdrift": None,
    # 같은 설정에서 E[sup_t X_t] 는? (sup X 의 분포 이름을 먼저 떠올리세요)
    "e_sup_negdrift": None,
    # T_1 = BM 이 1 에 처음 도달하는 시간: E[exp(-T_1 / 2)] 는?
    "laplace_T1": None,
}

## 4. Simulate — 시뮬레이션

먼저 이 노트북에서만 쓰는 도우미를 정의한다. `exit_times` 는 드리프트 BM 의 $(-a,b)$ 탈출을 경로에 대해 벡터화해 (시간에 대해서만 파이썬 루프) 시뮬레이션한다. `bridge=True` 이면 2.1 의 다리 교차 확률로 스텝 도중의 탈출을 잡아내고 탈출 시각을 스텝 중점으로 기록한다; `a=np.inf` 이면 아래 장벽이 없다 (E3). 반환값은 `(hit_b, tau, x_end)`: 위 장벽으로 나갔는지, 탈출 시각 (`t_max` 까지 못 나가면 `np.inf`), 그리고 미탈출 경로의 $t_{\max}$ 에서의 값.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def exit_times(a, b, mu, sigma, dt, n_paths, rng, t_max=40.0, bridge=True):
    """X_t = mu t + sigma B_t 의 (-a, b) 탈출. 경로 벡터화, 활성 경로만 갱신."""
    x = np.zeros(n_paths)
    tau = np.full(n_paths, np.inf)
    hit_b = np.zeros(n_paths, dtype=bool)
    active = np.ones(n_paths, dtype=bool)
    sd = sigma * np.sqrt(dt)
    for k in range(int(round(t_max / dt))):
        idx = np.flatnonzero(active)
        if idx.size == 0:
            break
        xo = x[idx]
        xn = xo + mu * dt + sd * rng.standard_normal(idx.size)
        up, dn = xn >= b, xn <= -a
        if bridge:  # 격자점은 안에 있어도 스텝 도중에 넘었을 확률 (브라운 다리 최댓값 CDF)
            p_up = np.exp(-2 * (b - xo) * (b - xn) / (sigma**2 * dt))
            p_dn = np.exp(-2 * (a + xo) * (a + xn) / (sigma**2 * dt)) if np.isfinite(a) else 0.0
            up |= (~dn) & (rng.random(idx.size) < p_up)
            dn |= (~up) & (rng.random(idx.size) < p_dn)
        done = up | dn
        tau[idx[done]] = (k + 0.5) * dt if bridge else (k + 1) * dt
        hit_b[idx[done]] = up[done]
        active[idx[done]] = False
        x[idx[~done]] = xn[~done]
    return hit_b, tau, x

`running_max_drift` 는 06c 의 다리 보정 러닝 최댓값을 드리프트 BM 으로 옮긴 것이다 (다리의 법칙은 $\mu$ 에 무관하므로 식이 같다). `exit_prob_drift`, `exit_time_drift` 는 정리 2·3 의 닫힌 형식이다.

In [ ]:
# helper (spkit에 없어서 여기서 정의)
def running_max_drift(mu, sigma, dt, n_steps, n_paths, rng):
    """드리프트 BM 의 다리 보정 러닝 최댓값 M (t_max = n_steps*dt) 과 끝값."""
    x = np.zeros(n_paths)
    M = np.zeros(n_paths)
    sd = sigma * np.sqrt(dt)
    for _ in range(n_steps):
        xn = x + mu * dt + sd * rng.standard_normal(n_paths)
        U = rng.random(n_paths)
        m_step = 0.5 * (x + xn + np.sqrt((xn - x) ** 2 - 2 * sigma**2 * dt * np.log(U)))
        M = np.maximum(M, m_step)
        x = xn
    return M, x


def exit_prob_drift(a, b, mu, sigma=1.0):
    """P(X_tau = b), X = mu t + sigma B, (-a, b) 탈출 (정리 2·3)."""
    if mu == 0:
        return a / (a + b)
    ea, eb = np.exp(2 * mu * a / sigma**2), np.exp(-2 * mu * b / sigma**2)
    return (ea - 1) / (ea - eb)


def exit_time_drift(a, b, mu, sigma=1.0):
    """E[tau] (정리 2·3)."""
    if mu == 0:
        return a * b / sigma**2
    p = exit_prob_drift(a, b, mu, sigma)
    return (b * p - a * (1 - p)) / mu

### 4.1 조건부 평균으로 마팅게일 검사 (05a 방식)

마팅게일 성질 $E[M_t\mid\mathcal F_s]=M_s$ 는 조건부 기대라 직접 보기 어렵다. 05a 에서 했듯이 $s=1$, $t=2$ 로 두고 $B_1$ 을 분위수 bin 으로 나눈 뒤, 각 bin 에서 $B_2$, $B_2^2-2$, $e^{B_2-1}$ 의 평균을 bin 중심 $x$ 에 대해 그리면 마팅게일 예측 $x$, $x^2-1$, $e^{x-1/2}$ 위에 올라타야 한다. (마르코프성 덕분에 $\mathcal F_1$ 대신 $B_1$ 만으로 조건을 걸어도 같다.)

In [ ]:
t_grid, B = brownian_paths(N_PATHS, 20, 2.0, rng)   # dt = 0.1: s = 1 → 열 10, t = 2 → 열 20
B1, B2 = B[:, 10], B[:, 20]
edges = np.quantile(B1, np.linspace(0, 1, N_BINS + 1))
which = np.clip(np.searchsorted(edges, B1, side="right") - 1, 0, N_BINS - 1)
centers = np.array([B1[which == j].mean() for j in range(N_BINS)])
targets = {"E[B_2 | B_1]": (B2, lambda x: x),
           "E[B_2^2 - 2 | B_1]": (B2**2 - 2, lambda x: x**2 - 1),
           "E[exp(B_2 - 1) | B_1]": (np.exp(B2 - 1), lambda x: np.exp(x - 0.5))}

fig, axes = plt.subplots(1, 3, figsize=(12, 3.6))
xx = np.linspace(centers[0], centers[-1], 200)
for ax, (name, (Y, pred)) in zip(axes, targets.items()):
    means = np.array([Y[which == j].mean() for j in range(N_BINS)])
    ax.plot(centers, means, "o", label="binned mean")
    ax.plot(xx, pred(xx), "-", label="martingale prediction")
    ax.set_xlabel("$B_1$ (bin center)"); ax.set_title(name)
axes[0].legend(); plt.tight_layout(); plt.show()

조건 없는 검사도 해 둔다: $E[B_1^2-1]=0$, $E[e^{B_1-1/2}]=1$ (Compare 에서 assert). 지수 마팅게일의 표본 표준편차 $\sqrt{e-1}\approx1.31$ 이 $B_1^2-1$ 의 $\sqrt2$ 와 비슷한 크기임을 눈여겨보라 — $t=5$ 였다면 $\sqrt{e^5-1}\approx12$ 다 (함정 6).

In [ ]:
m2_hat = mc_estimate(B1**2 - 1)
exp_mart_hat = mc_estimate(np.exp(B1 - 0.5))
print(f"E[B_1^2 - 1]      = {m2_hat}   (exact 0)")
print(f"E[exp(B_1 - 1/2)] = {exp_mart_hat}   (exact 1; sd = sqrt(e-1) = {np.sqrt(np.e - 1):.3f})")

### 4.2 구간 탈출 ($\mu=0$)

$a=1$, $b=2$, $\Delta t=0.005$, 다리 보정 켬. 경로당 $P(\tau>40)\approx3\times10^{-10}$ 이므로 미탈출 경로는 없어야 한다.

In [ ]:
A, BB = 1.0, 2.0
hit_b, tau, _ = exit_times(A, BB, 0.0, 1.0, 0.005, N_PATHS, rng, t_max=40.0, bridge=True)
p_exit_hat, e_tau_hat = mc_proportion(hit_b), mc_estimate(tau)
print(f"P(B_tau = 2) = {p_exit_hat}   (exact {A/(A+BB):.4f})")
print(f"E[tau]       = {e_tau_hat}   (exact {A*BB:.4f});  max tau = {tau.max():.2f}, not exited = {np.isinf(tau).sum()}")

왼쪽: BM 5 경로와 두 장벽 (격자 위 탈출점 표시, 탈출 후는 흐리게). 가운데: $\tau$ 의 히스토그램 — 평균 2 지만 오른쪽으로 긴 꼬리. 오른쪽: semilog-y 경험 꼬리 $P(\tau>t)$ 와 기준 기울기 $-\lambda_1=-\pi^2/(2\cdot3^2)\approx-0.548$. 꼬리가 결국 직선(지수)이 되는 것은 06c 의 $t^{-1/2}$ 멱꼬리(한쪽 장벽)와 대조적이다.

In [ ]:
lambda_1 = np.pi**2 / (2 * (A + BB) ** 2)
t5, B5 = brownian_paths(5, 4000, 20.0, rng)
fig, axes = plt.subplots(1, 3, figsize=(13, 3.8))
for row in B5:
    out = np.flatnonzero((row >= BB) | (row <= -A))
    k = out[0] if out.size else row.size - 1
    (line,) = axes[0].plot(t5[: k + 1], row[: k + 1], lw=1)
    axes[0].plot(t5[k:], row[k:], lw=0.6, alpha=0.25, color=line.get_color())
    axes[0].plot(t5[k], row[k], "o", color=line.get_color(), ms=5)
axes[0].axhline(BB, color="k", ls="--"); axes[0].axhline(-A, color="k", ls="--")
axes[0].set_xlabel("t"); axes[0].set_title("5 paths, barriers -1 and 2")
axes[1].hist(tau, bins=60, density=True); axes[1].axvline(tau.mean(), color="C3", label=f"mean = {tau.mean():.3f}")
axes[1].set_xlabel("exit time tau"); axes[1].set_ylabel("density"); axes[1].legend()
ts = np.sort(tau); surv = 1 - np.arange(1, ts.size + 1) / ts.size
axes[2].semilogy(ts[:-1], surv[:-1], lw=1, label="empirical P(tau > t)")
t0 = 3.0; s0 = (tau > t0).mean()
axes[2].semilogy([t0, ts[-1]], [s0, s0 * np.exp(-lambda_1 * (ts[-1] - t0))], "k--", label=f"slope -pi^2/18 = {-lambda_1:.3f}")
axes[2].set_xlabel("t"); axes[2].set_title("tail of tau (semilog-y)"); axes[2].legend()
plt.tight_layout(); plt.show()

### 4.3 이산화 편향 — 격자 vs 다리 보정

같은 문제를 다리 보정 **없이** $\Delta t\in\{0.05,0.005\}$ 에서, 그리고 보정을 켜고 성긴 $\Delta t=0.05$ 에서 돌린다. 06c 의 밀린 장벽 그림에 따르면 격자 탈출은 장벽이 $\delta=0.5826\sqrt{\Delta t}$ 씩 밖으로 밀린 문제와 같아서 $\hat p\approx(a+\delta)/(a+b+2\delta)$, $\hat E\tau\approx(a+\delta)(b+\delta)$ 가 나와야 한다.

In [ ]:
BETA = 0.5826   # -zeta(1/2)/sqrt(2 pi): 06c 의 격자 편향 상수
bias_rows, bias_res = [], {}
for dt, bridge in [(0.05, False), (0.005, False), (0.05, True)]:
    h, ta, _ = exit_times(A, BB, 0.0, 1.0, dt, N_PATHS, rng, t_max=40.0, bridge=bridge)
    d = 0.0 if bridge else BETA * np.sqrt(dt)
    ph, eh = mc_proportion(h), mc_estimate(ta)
    bias_res[(dt, bridge)] = (ph, eh)
    bias_rows.append(f"| {'bridge' if bridge else 'grid'} | {dt} | {ph.mean:.4f} ± {ph.se:.4f} | {(A+d)/(A+BB+2*d):.4f} "
                     f"| {eh.mean:.4f} ± {eh.se:.4f} | {(A+d)*(BB+d):.4f} | {A/(A+BB):.4f} / {A*BB:.1f} |")
Markdown("| method | dt | p_hat | shifted-barrier pred. | E[tau]_hat | shifted-barrier pred. | true |\n|---|---|---|---|---|---|---|\n"
         + "\n".join(bias_rows))

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 3.8))
for bridge, marker, lab in [(False, "s", "grid only"), (True, "o", "bridge-corrected")]:
    pts = [(dt, r[1]) for (dt, br), r in bias_res.items() if br == bridge]
    if bridge:
        pts.append((0.005, e_tau_hat))
    pts.sort()
    ax.errorbar([p[0] for p in pts], [p[1].mean for p in pts], yerr=[1.96 * p[1].se for p in pts],
                fmt=marker + "-", capsize=3, label=lab)
dts = np.linspace(0.002, 0.06, 100)
ax.plot(dts, (A + BETA * np.sqrt(dts)) * (BB + BETA * np.sqrt(dts)), "k:", label="(a + 0.58 sqrt(dt))(b + 0.58 sqrt(dt))")
ax.axhline(A * BB, color="k", ls="--", label="ab = 2")
ax.set_xlabel("dt"); ax.set_ylabel("E[tau] estimate"); ax.set_title("grid bias of the exit time"); ax.legend(loc="upper left")
plt.show()

격자 편향은 $\sqrt{\Delta t}$ 로만 줄어서 $\Delta t$ 를 10 배 줄여도 편향은 $\sqrt{10}\approx3.2$ 배만 준다. 다리 보정은 $\Delta t=0.05$ 에서도 참값 위에 있다 (중점 반올림 $\le\Delta t/2=0.025\approx2$ SE 라 이 행은 표에만 보이고 assert 하지 않는다).

### 4.4 드리프트 탈출

$\mu=0.25$ 에서 정리 3 의 $p=0.5065$, $E\tau=2.078$ 을 확인하고, $\mu\in\{-1,-0.5,0.5,1\}$ 에서 $\hat p(\mu)$ 를 정확 곡선 위에 올린다.

In [ ]:
MU = 0.25
hit_d, tau_d, _ = exit_times(A, BB, MU, 1.0, 0.005, N_PATHS, rng, t_max=40.0, bridge=True)
p_exit_drift_hat, e_tau_drift_hat = mc_proportion(hit_d), mc_estimate(tau_d)
print(f"mu = {MU}: P(X_tau = 2) = {p_exit_drift_hat}   (exact {exit_prob_drift(A, BB, MU):.6f})")
print(f"          E[tau]       = {e_tau_drift_hat}   (exact {exit_time_drift(A, BB, MU):.6f}); not exited = {np.isinf(tau_d).sum()}")

mu_list = [-1.0, -0.5, 0.5, 1.0]
curve_hat = [mc_proportion(exit_times(A, BB, mu, 1.0, 0.005, N_CURVE, rng, 40.0, True)[0]) for mu in mu_list]
for mu, r in zip(mu_list, curve_hat):
    print(f"mu = {mu:+.1f}: p_hat = {r}   exact {exit_prob_drift(A, BB, mu):.5f}")

In [ ]:
mus = np.linspace(-1.5, 1.5, 301)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(mus, [exit_prob_drift(A, BB, m) for m in mus], "-", label="exact $(e^{2\\mu a}-1)/(e^{2\\mu a}-e^{-2\\mu b})$")
ax.errorbar(mu_list + [MU], [r.mean for r in curve_hat] + [p_exit_drift_hat.mean],
            yerr=[1.96 * r.se for r in curve_hat] + [1.96 * p_exit_drift_hat.se], fmt="o", capsize=3, label="MC (95% CI)")
ax.axhline(A / (A + BB), color="k", ls="--", label="a/(a+b) = 1/3")
ax.set_xlabel("drift mu"); ax.set_ylabel("P(exit at b = 2 before -a = -1)"); ax.legend()
plt.show()

### 4.5 한쪽 장벽, 음의 드리프트: $\sup X\sim\mathrm{Exp}(2|\mu|/\sigma^2)$

$\mu=-0.5$, $\sigma=1$, $b=1$. 러닝 최댓값 $M=\max_{t\le40}X_t$ 를 다리 보정으로 구하면 $P(M\ge1)\approx e^{-1}$, $E[M]\approx1/(2|\mu|)=1$. 지평 40 의 유한 지평 정확값은 $P(M_{40}\ge1)=0.367839$, $E[M_{40}]=0.999781$ (아래 코드에서 06e 의 드리프트 BM 최댓값 분포로 계산) 로 무한 지평 값과의 차이가 se 보다 훨씬 작다.

In [ ]:
MU1, SIG1, B1LVL, T_HOR = -0.5, 1.0, 1.0, 40.0
M, _ = running_max_drift(MU1, SIG1, 0.01, int(T_HOR / 0.01), N_PATHS, rng)
p_sup_hat, e_sup_hat = mc_proportion(M >= B1LVL), mc_estimate(M)
# 유한 지평 정확값 (06e 예고): P(M_T >= x) = Phi_bar((x - mu T)/sqrt T) + e^{2 mu x} Phi_bar((x + mu T)/sqrt T)
surv_T = lambda x: norm.sf((x - MU1 * T_HOR) / np.sqrt(T_HOR)) + np.exp(2 * MU1 * x / SIG1**2) * norm.sf((x + MU1 * T_HOR) / np.sqrt(T_HOR))
p_sup_finite, e_sup_finite = surv_T(B1LVL), quad(surv_T, 0, 60)[0]
print(f"P(sup X >= 1) = {p_sup_hat}   (exact e^-1 = {np.exp(2*MU1*B1LVL):.6f}; horizon-40 exact {p_sup_finite:.6f})")
print(f"E[sup X]      = {e_sup_hat}   (exact 1/(2|mu|) = {1/(2*abs(MU1)):.6f}; horizon-40 exact {e_sup_finite:.6f})")

rate = 2 * abs(MU1) / SIG1**2
xg = np.linspace(0, 6, 300)
ax = plot_hist_vs_pdf(M, xg, rate * np.exp(-rate * xg), bins=60, label_pdf="Exp(1) pdf")
ax.axvline(B1LVL, color="C3", ls="--", label=f"x = 1: P = e^-1 = {np.exp(-1):.3f}")
ax.set_xlabel("sup of -0.5 t + B_t over [0, 40]"); ax.legend(); plt.show()

### 4.6 라플라스 변환 $E[e^{-\alpha T_1}]=e^{-\sqrt{2\alpha}}$

06c 의 정확 표본기 $T_1\overset{d}{=}1/Z^2$ ($Z\sim N(0,1)$) 를 쓰면 시뮬레이션 없이 라플라스 변환을 검사할 수 있다. $\alpha\downarrow0$ 에서 곡선의 기울기가 $-\infty$ 인 것이 $E[T_1]=\infty$ 다.

In [ ]:
T1 = 1.0 / rng.standard_normal(N_LAPLACE) ** 2
laplace_hat = mc_estimate(np.exp(-T1 / 2))
print(f"E[exp(-T_1/2)] = {laplace_hat}   (exact e^-1 = {np.exp(-1):.6f})")
alphas = np.linspace(0, 3, 13)
lap_mc = [mc_estimate(np.exp(-al * T1)) for al in alphas]
ag = np.linspace(0, 3, 300)
fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.plot(ag, np.exp(-np.sqrt(2 * ag)), "-", label="exact $e^{-\\sqrt{2\\alpha}}$")
ax.errorbar(alphas, [r.mean for r in lap_mc], yerr=[1.96 * r.se for r in lap_mc], fmt="o", capsize=3, label="MC, T_1 = 1/Z^2")
ax.set_xlabel("alpha"); ax.set_ylabel("E[exp(-alpha T_1)]"); ax.set_title("slope -inf at alpha = 0  <=>  E[T_1] = inf"); ax.legend()
plt.show()

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

닫힌 형식은 전부 정리 1–3 의 식으로 코드에서 계산한다. 격자 전용(grid only) 세 행은 편향 시연이라 'investigate' 가 **나와야 정상**이고, 다리 $\Delta t=0.05$ 의 $E\tau$ 행은 중점 반올림 오차($\le\Delta t/2$) 가 se 와 같은 크기라 표에만 보인다.

In [ ]:
p_exact, et_exact = A / (A + BB), A * BB
p_drift_exact, et_drift_exact = exit_prob_drift(A, BB, MU), exit_time_drift(A, BB, MU)
p_sup_exact, e_sup_exact = np.exp(2 * MU1 * B1LVL / SIG1**2), SIG1**2 / (2 * abs(MU1))
laplace_exact = np.exp(-1.0 * np.sqrt(2 * 0.5))
P = predictions
rows = [
    {"name": "E[B_1^2 - 1]", "predicted": None, "estimate": m2_hat, "exact": 0.0},
    {"name": "E[exp(B_1 - 1/2)]", "predicted": None, "estimate": exp_mart_hat, "exact": 1.0},
    {"name": "P(B_tau = 2), bridge, dt = 0.005", "predicted": P["p_hit_b_first"], "estimate": p_exit_hat, "exact": p_exact},
    {"name": "E[tau], bridge, dt = 0.005", "predicted": P["e_exit_time"], "estimate": e_tau_hat, "exact": et_exact},
    {"name": "P(B_tau = 2), grid only, dt = 0.05 [expect investigate]", "predicted": None, "estimate": bias_res[(0.05, False)][0], "exact": p_exact},
    {"name": "E[tau], grid only, dt = 0.05 [expect investigate]", "predicted": None, "estimate": bias_res[(0.05, False)][1], "exact": et_exact},
    {"name": "E[tau], grid only, dt = 0.005 [expect investigate]", "predicted": None, "estimate": bias_res[(0.005, False)][1], "exact": et_exact},
    {"name": "E[tau], bridge, dt = 0.05 (midpoint error <= 0.025)", "predicted": None, "estimate": bias_res[(0.05, True)][1], "exact": et_exact},
    {"name": "P(X_tau = 2), mu = 0.25", "predicted": P["p_hit_b_first_drift"], "estimate": p_exit_drift_hat, "exact": p_drift_exact},
    {"name": "E[tau], mu = 0.25", "predicted": P["e_exit_time_drift"], "estimate": e_tau_drift_hat, "exact": et_drift_exact},
    {"name": "P(sup X >= 1), mu = -0.5 (horizon 40)", "predicted": P["p_ever_hit_1_negdrift"], "estimate": p_sup_hat, "exact": p_sup_exact},
    {"name": "E[sup X], mu = -0.5 (horizon 40)", "predicted": P["e_sup_negdrift"], "estimate": e_sup_hat, "exact": e_sup_exact},
    {"name": "E[exp(-T_1/2)]", "predicted": P["laplace_T1"], "estimate": laplace_hat, "exact": laplace_exact},
]
Markdown(compare_table(rows))

각 행에서 불일치가 뜻하는 것:
- **E[B_1^2 - 1], E[exp(B_1 - 1/2)]**: 증분 분산이 $\Delta t$ 가 아니거나 (`brownian_paths` 오류), 지수 마팅게일의 시간항 $\theta^2t/2$ 를 잘못 썼다.
- **P(B_tau = 2), E[tau] (bridge, dt = 0.005)**: 다리 교차 확률의 식(부호, $\sigma^2\Delta t$)이나 `up`/`dn` 의 배타 처리가 틀렸거나, 정리 2 의 $a/(a+b)$, $ab$ 를 잘못 대응시켰다 ($a$ 는 아래 장벽까지의 거리!).
- **grid only 세 행**: 'investigate' 가 **안** 나오면 오히려 이상하다 — 격자 편향이 사라졌다면 `bridge` 플래그가 꼬였을 가능성.
- **E[tau], bridge, dt = 0.05**: 2–3 SE 벗어나도 중점 반올림 때문일 수 있다; 6 SE 이상이면 보정식을 의심.
- **P(X_tau = 2), E[tau] (mu = 0.25)**: 지수 마팅게일의 $\theta=-2\mu/\sigma^2$ 부호, 또는 월드 항등식 $E[X_\tau]=\mu E\tau$ 의 부호가 뒤집혔다 (함정 3 의 검산: $\mu>0$ 이면 $p>1/3$).
- **P(sup X >= 1), E[sup X]**: 러닝 최댓값의 다리 보정을 빼먹었거나(격자 최댓값은 작게 나온다), $\mathrm{Exp}$ 의 모수 $2|\mu|/\sigma^2$ 를 $|\mu|$ 로 착각했다.
- **E[exp(-T_1/2)]**: $T_1=1/Z^2$ 표본기 자체(06c) 또는 $\alpha=\theta^2/2$ 의 대응이 틀렸다.

In [ ]:
assert_close(m2_hat, 0.0, k=4, name="E[B_1^2-1]")
assert_close(exp_mart_hat, 1.0, k=4, name="E[exp(B_1-1/2)]")
assert_close(p_exit_hat, p_exact, k=4, name="P(B_tau=2) bridge dt=0.005")
assert_close(e_tau_hat, et_exact, k=4, name="E[tau] bridge dt=0.005")
assert not np.isinf(tau).any(), "all paths must exit before t_max = 40"
assert_close(p_exit_drift_hat, p_drift_exact, k=4, name="P(X_tau=2) mu=0.25")
assert_close(e_tau_drift_hat, et_drift_exact, k=4, name="E[tau] mu=0.25")
assert_close(p_sup_hat, p_sup_exact, k=4, name="P(sup X>=1) mu=-0.5")
assert_close(e_sup_hat, e_sup_exact, k=4, name="E[sup X] mu=-0.5")
assert_close(laplace_hat, laplace_exact, k=4, name="E[exp(-T_1/2)]")
# 격자 전용 행과 다리 dt=0.05 행은 assert 하지 않고 z-score 만 출력한다 (편향 시연 / 중점 반올림)
for (dt, br), (ph, eh) in bias_res.items():
    print(f"{'bridge' if br else 'grid  '} dt={dt}: z(p) = {(ph.mean - p_exact)/ph.se:+.1f} SE,  z(E tau) = {(eh.mean - et_exact)/eh.se:+.1f} SE")
print("all asserts passed")

## 6. 연습문제

### E1 (계산) — 정리 2·3 을 손으로

(a) 구간 $(-3,1)$, $B_0=0$: 1 을 먼저 칠 확률과 $E\tau$. (b) 같은 구간에서 $\sigma=2$ 이면? (c) 구간 $(-1,2)$, 시작점 $x=0.5$: $P_x(B_\tau=2)$ 와 $E_x\tau$. (d) 05b 의 $p\ne1/2$ 파산 확률 $\dfrac{1-(q/p)^i}{1-(q/p)^N}$ 이 오늘의 드리프트 공식과 같은 꼴임을 보여라 ($e^{-2\mu}\leftrightarrow q/p$).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) $a=3$, $b=1$: $P=a/(a+b)=3/4$, $E\tau=ab=3$. (b) 확률은 $\sigma$ 와 무관하게 $3/4$; $E\tau=ab/\sigma^2=3/4$. (c) $a=1$, $b=2$, $x=0.5$: $P_x=(x+a)/(a+b)=1.5/3=0.5$, $E_x\tau=(x+a)(b-x)=1.5\times1.5=2.25$.

(d) 05b 에서 $(q/p)^{S_n}$ 이 마팅게일인 것은 $E[(q/p)^{\pm1}]=p\cdot(q/p)+q\cdot(p/q)=q+p=1$ 때문이다. 오늘은 $e^{-2\mu X_t/\sigma^2}$ 가 마팅게일 — 둘 다 "위로 가면 작아지고 아래로 가면 커지는 환율"이다. 시작점 $i$, 장벽 $0,N$ 에서 $1=P(\text{win})(q/p)^N+(1-P(\text{win}))(q/p)^0$ 를 $i$ 에서 시작하도록 $(q/p)^i$ 로 정규화하면 $P(\text{win})=\dfrac{1-(q/p)^i}{1-(q/p)^N}$; 오늘 식에서 $x=0$, 장벽 $-a,b$ 를 $i\leftrightarrow a$, $N\leftrightarrow a+b$, $(q/p)\leftrightarrow e^{-2\mu/\sigma^2}$ 로 두면 $\dfrac{1-e^{2\mu a}}{e^{-2\mu b}-e^{2\mu a}}=\dfrac{e^{-2\mu a}-1}{e^{-2\mu(a+b)}-1}=\dfrac{1-(q/p)^{i}}{1-(q/p)^{N}}$ 로 정확히 같은 꼴이다.

```python
print(3/4, 3*1, 3*1/2**2, (0.5+1)/3, (0.5+1)*(2-0.5))
# 0.75 3 0.75 0.5 2.25
```

</details>

### E2 (유도 후 시뮬레이션 검증) — 월드 항등식의 연속판

$X_t-\mu t=\sigma B_t$ 가 마팅게일임을 이용해 드리프트 BM 의 $E\tau=(bp-a(1-p))/\mu$ 를 유도하라. 그리고 $\mu=-0.25$, $\sigma=1$, $a=1$, $b=2$ 에서 $p$ 와 $E\tau$ 를 계산하고 (부호 검산: $\mu<0$ 이면 $p<1/3$ 이어야 한다) `exit_times(1, 2, -0.25, 1, 0.005, N_PATHS, rng, 40, True)` 로 `assert_close` 하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $\sigma B_{t\wedge\tau}=X_{t\wedge\tau}-\mu(t\wedge\tau)$ 는 유계 정지시간 $t\wedge\tau$ 에서 평균 0: $E[X_{t\wedge\tau}]=\mu E[t\wedge\tau]$. $t\to\infty$ 에서 좌변은 유계수렴($|X_{t\wedge\tau}|\le\max(a,b)$), 우변은 단조수렴으로 $E[X_\tau]=\mu E\tau$. $X_\tau\in\{b,-a\}$ 이므로 $bp-a(1-p)=\mu E\tau$. (이것이 월드 항등식 $E[S_\tau]=E[\xi]E[\tau]$ 의 연속판이다.)

**수치.** $p=\dfrac{e^{-0.5}-1}{e^{-0.5}-e^{1}}=0.186324<1/3$ ✓, $E\tau=\dfrac{2(0.186324)-0.813676}{-0.25}=1.764115$.

```python
mu_e2 = -0.25
p_e2, et_e2 = exit_prob_drift(1, 2, mu_e2), exit_time_drift(1, 2, mu_e2)
h_e2, ta_e2, _ = exit_times(1, 2, mu_e2, 1.0, 0.005, N_PATHS, rng, 40.0, True)
print(p_e2, et_e2)                                  # 0.186324  1.764115
print(mc_proportion(h_e2), mc_estimate(ta_e2))      # 약 0.184 ± 0.003, 1.76 ± 0.011 (N=20000)
assert_close(mc_proportion(h_e2), p_e2, k=4, name="E2 p")
assert_close(mc_estimate(ta_e2), et_e2, k=4, name="E2 E[tau]")
```

</details>

### E3 (가정을 깨보기) — 선택정지의 조건

한쪽 장벽 $T_1$ ($\mu=0$) 에 대해 $B_{T_1}=1$ 이므로 $E[B_{T_1}]=1\ne0$. 그러나 유계 정지시간 $T\wedge T_1$ 에서는 $E[B_{T\wedge T_1}]=0$ 이 정확히 성립한다. `exit_times(np.inf, 1, 0, 1, 0.001, N_PATHS, rng, t_max=T, bridge=True)` 를 $T\in\{1,4,16\}$ 로 돌려 (`x_end` 는 미탈출 경로의 $B_T$) `mc_estimate(np.where(hit_b, 1.0, x_end))` 가 0 임을, 동시에 `mc_proportion(hit_b)` 가 $2(1-\Phi(1/\sqrt T))=0.3173,\ 0.6171,\ 0.8026$ 으로 1 에 다가감을 확인하라. 미탈출 경로의 평균 $E[B_T\mid T_1>T]$ 는 어디로 가는가?

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

$T\wedge T_1\le T$ 는 유계 정지시간이므로 선택정지 (i) 로 $E[B_{T\wedge T_1}]=0$ — 모든 $T$ 에서 정확히. 이를 쪼개면 $0=1\cdot P(T_1\le T)+E[B_T;\,T_1>T]$ 이므로
$$E[B_T\mid T_1>T]=-\frac{P(T_1\le T)}{P(T_1>T)}:\quad T=1:\ -0.465,\quad T=4:\ -1.612,\quad T=16:\ -4.07\ \to\ -\infty.$$
즉 도달한 경로의 $+1$ 을 상쇄하려고 아직 도달 못 한 소수의 경로가 점점 더 깊이 내려가 있다. $T\to\infty$ 에서 $B_{T\wedge T_1}\to B_{T_1}=1$ a.s. 지만 기댓값은 0 에 머문다: **극한과 기댓값의 교환이 실패**한다 (지배수렴 불가 — $E[\sup_{t\le T_1}|B_t|]=\infty$; 균등적분가능도 아니다). 선택정지 조건 (ii)/(iii) 가 정확히 이 교환을 보장하는 조건이다.

se 크기: $\mathrm{Var}(B_{T\wedge T_1})=E[T\wedge T_1]=0.849,\ 2.323,\ 5.449$ ($T=1,4,16$) 이므로 $N=20000$ 에서 se $\approx0.0065,\ 0.011,\ 0.0165$ — 검사의 해상도는 $T$ 가 클수록 나빠진다.

```python
for T in [1, 4, 16]:
    h, ta, xe = exit_times(np.inf, 1.0, 0.0, 1.0, 0.001, N_PATHS, rng, t_max=T, bridge=True)
    stopped = mc_estimate(np.where(h, 1.0, xe))            # E[B_{T ∧ T_1}] → 0
    p_hit = mc_proportion(h)                                # → 2(1 - Phi(1/sqrt T))
    print(T, stopped, p_hit, f"exact {2*norm.sf(1/np.sqrt(T)):.4f}",
          f"E[B_T | T_1 > T] = {xe[~h].mean():.3f} (exact {-p_hit.mean/(1-p_hit.mean):.3f})")
    assert_close(stopped, 0.0, k=4, name=f"E[B_(T∧T_1)], T={T}")
    assert_close(p_hit, 2 * norm.sf(1 / np.sqrt(T)), k=4, name=f"P(T_1<=T), T={T}")
```

</details>

## 7. 정리

1. $B_t$, $B_t^2-t$, $e^{\theta B_t-\theta^2t/2}$ 는 마팅게일 — 독립 증분 + 정규 mgf 가 전부다. 드리프트 $X=\mu t+\sigma B$ 에는 $e^{-2\mu X_t/\sigma^2}$.
2. $(-a,b)$ 탈출: $P(B_\tau=b)=a/(a+b)$, $E\tau=ab/\sigma^2$ — 유계 정지 마팅게일에 선택정지 (도박꾼 파산의 4 번째 풀이; 05b 의 $i/N$, $i(N-i)$ 와 같은 꼴).
3. 드리프트: $P(X_\tau=b)=\dfrac{e^{2\mu a/\sigma^2}-1}{e^{2\mu a/\sigma^2}-e^{-2\mu b/\sigma^2}}$, $E\tau=(bp-a(1-p))/\mu$; 한쪽 장벽 $P(T_b<\infty)=e^{-2|\mu|b/\sigma^2}$, $\sup X\sim\mathrm{Exp}(2|\mu|/\sigma^2)$; 라플라스 $E[e^{-\alpha T_b}]=e^{-b\sqrt{2\alpha}}$.
4. 선택정지의 조건은 장식이 아니다: 한쪽 장벽 $T_b$ 에서는 $E[B_{T_b}]=b\ne0$ — 유계 $T\wedge T_b$ 에서는 0 이 맞고 $T\to\infty$ 의 교환이 실패한다.
5. 탈출 시뮬레이션은 격자 편향(장벽이 $0.58\sqrt{\Delta t}$ 밀림)을 갖는다; 다리 교차 확률 $e^{-2(b-x)(b-y)/(\sigma^2\Delta t)}$ 로 보정하면 확률은 정확, 시각은 $\Delta t/2$ 이내.

**Trap 카드.** Q: $B_t$ 가 마팅게일이고 $T_1<\infty$ a.s. 이면 $E[B_{T_1}]=0$ 아닌가? → A: 아니다. $B_{T_1}=1$ 이므로 $E[B_{T_1}]=1$. 선택정지의 조건($ET_1<\infty$, 또는 정지 과정의 유계성/균등적분가능성)이 깨졌다. 유계 정지 $T\wedge T_1$ 에서는 $E=0$ 이 맞고, $T\to\infty$ 에서 극한과 기댓값의 교환이 실패한다.

**다음 노트북: 06e** — 오늘 나온 세 대상, 즉 격자 사이를 채운 브라운 다리, 지수 마팅게일이 다룬 드리프트 BM, 그리고 $e^{\theta B_t-\theta^2t/2}$ 자체(정규화하면 기하 브라운 운동) 를 06e 에서 각각 하나의 과정으로 정식화한다: 다리의 공분산 $\min(s,t)-st/T$, 드리프트 BM 의 최댓값 분포(오늘 4.5 의 유한 지평 공식), GBM 의 로그정규 모멘트.

log/progress.md 한 줄 템플릿: `- [ ] 06d 브라운 마팅게일과 탈출 문제 — YYYY-MM-DD, 소요 __분, Predict 적중 __/7, 막힌 곳: __`